In [2]:
!pip install promptbench

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 622.8/622.8 kB 14.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 73.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.6/57.6 kB 4.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.1/131.1 kB 11.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 4.7 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of datasets to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of datasets to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking

In [ ]:
import os

# Some Kaggle images ship a TensorFlow install that conflicts with protobuf,
# which crashes `transformers` when it lazily checks for TF tensors during
# generation. We only use PyTorch here, so disable the TF/Flax code paths.
os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"


In [4]:
import promptbench as pb

In [ ]:
from promptbench.metrics.eval import Eval

# Valid label classes for this task. Any prediction outside this set (i.e.
# mapped to -1 by the projection function when the model output can't be
# parsed into a label) is scored as incorrect, rather than being excluded
# or treated as its own separate class.
RTE_CLASSES = [0, 1]
RTE_CLASS_NAMES = {0: "entailment", 1: "contradiction"}


def f1_score_manual(y_true, y_pred, classes=None, average=None):
    """Macro-averaged F1 over a fixed set of valid classes, so an unmapped
    (-1) prediction always counts as wrong instead of forming its own
    phantom class in the average."""
    if classes is None:
        classes = sorted(set(y_true))

    f1s = []
    for label in classes:
        tp = sum(1 for yt, yp in zip(y_true, y_pred) if yt == label and yp == label)
        fp = sum(1 for yt, yp in zip(y_true, y_pred) if yt != label and yp == label)
        fn = sum(1 for yt, yp in zip(y_true, y_pred) if yt == label and yp != label)

        precision = tp / (tp + fp) if tp + fp > 0 else 0
        recall = tp / (tp + fn) if tp + fn > 0 else 0
        f1 = 2 * precision * recall / (precision + recall) if precision + recall > 0 else 0
        f1s.append(f1)
    return sum(f1s) / len(f1s)


def confusion_matrix_manual(y_true, y_pred, classes):
    """Pure-Python confusion matrix (no pandas/numpy/sklearn dependency, to
    avoid environment issues seen on some Kaggle images). Rows are the
    fixed `classes`; columns add an "unmapped(-1)" bucket if needed."""
    has_unmapped = any(yp not in classes for yp in y_pred)
    columns = list(classes) + (["unmapped(-1)"] if has_unmapped else [])

    cm = {label: {col: 0 for col in columns} for label in classes}
    for yt, yp in zip(y_true, y_pred):
        if yt not in cm:
            continue
        col = yp if yp in classes else "unmapped(-1)"
        cm[yt][col] += 1
    return cm, columns


def classwise_report(y_true, y_pred, classes, class_names=None):
    """Per-class precision/recall/F1/support, the confusion matrix, and
    the unmapped (-1) count/rate."""
    rows = []
    for label in classes:
        tp = sum(1 for yt, yp in zip(y_true, y_pred) if yt == label and yp == label)
        fp = sum(1 for yt, yp in zip(y_true, y_pred) if yt != label and yp == label)
        fn = sum(1 for yt, yp in zip(y_true, y_pred) if yt == label and yp != label)
        precision = tp / (tp + fp) if tp + fp > 0 else 0.0
        recall = tp / (tp + fn) if tp + fn > 0 else 0.0
        f1 = 2 * precision * recall / (precision + recall) if precision + recall > 0 else 0.0
        support = sum(1 for yt in y_true if yt == label)
        name = class_names.get(label, str(label)) if class_names else str(label)
        rows.append({
            "class": name, "precision": precision, "recall": recall,
            "f1": f1, "support": support,
        })

    n_unmapped = sum(1 for yp in y_pred if yp not in classes)
    unmapped_rate = n_unmapped / len(y_pred) if len(y_pred) > 0 else 0.0

    cm, cm_columns = confusion_matrix_manual(y_true, y_pred, classes)

    return rows, cm, cm_columns, n_unmapped, unmapped_rate


def print_classwise_report(rows):
    header = f"{'class':<14}{'precision':>10}{'recall':>10}{'f1':>10}{'support':>10}"
    print(header)
    print("-" * len(header))
    for r in rows:
        print(f"{r['class']:<14}{r['precision']:>10.3f}{r['recall']:>10.3f}{r['f1']:>10.3f}{r['support']:>10d}")


def print_confusion_matrix(cm, cm_columns, classes, class_names=None):
    col_names = [class_names.get(c, str(c)) if class_names and c in classes else str(c) for c in cm_columns]
    row_names = [class_names.get(c, str(c)) if class_names else str(c) for c in classes]
    colw = max(12, max(len(c) for c in col_names) + 2)
    rowlabelw = max(len(r) for r in row_names) + 2

    header = " " * rowlabelw + "".join(f"{c:>{colw}}" for c in col_names)
    print(header)
    for label, rname in zip(classes, row_names):
        line = f"{rname:<{rowlabelw}}" + "".join(f"{cm[label][col]:>{colw}d}" for col in cm_columns)
        print(line)


Eval.compute_f1 = staticmethod(f1_score_manual)


In [6]:
dataset = pb.DatasetLoader.load_dataset("rte")
dataset[:5]

Generating train split:   0%|          | 0/2490 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/277 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3000 [00:00<?, ? examples/s]

[{'content': 'Sentence 1: Dana Reeve, the widow of the actor Christopher Reeve, has died of lung cancer at age 44, according to the Christopher Reeve Foundation. Sentence 2: Christopher Reeve had an accident.',
  'label': 1},
 {'content': 'Sentence 1: Yet, we now are discovering that antibiotics are losing their effectiveness against illness. Disease-causing bacteria are mutating faster than we can come up with new antibiotics to fight the new variations. Sentence 2: Bacteria is winning the war against antibiotics.',
  'label': 0},
 {'content': 'Sentence 1: Cairo is now home to some 15 million people - a burgeoning population that produces approximately 10,000 tonnes of rubbish per day, putting an enormous strain on public services. In the past 10 years, the government has tried hard to encourage private investment in the refuse sector, but some estimate 4,000 tonnes of waste is left behind every day, festering in the heat as it waits for someone to clear it up. It is often the people 

In [ ]:
counts = Counter(item['label'] for item in dataset)
print("Entailment:", counts[0])
print("Contradiction:", counts[1])


In [8]:
model = pb.LLMModel(model='google/flan-t5-large', max_new_tokens=50, temperature=0.3, device='cuda')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


config.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.13G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [ ]:
prompts = pb.Prompt([
    # Original / baseline
    "Given the following premise and hypothesis, determine their relationship. You must answer with exactly one label: entailment or contradiction.\n\n{content}\nAnswer:",

    # Rewording
    "Read the premise and hypothesis below and decide whether the hypothesis is supported by or contradicts the premise. Answer using exactly one label: entailment or contradiction.\n\n{content}\nLabel:",

    # Style change (formal / casual)
    "Analyze the logical relation between the statements below. Respond with only one of the following: entailment or contradiction.\n\n{content}\nResponse:",
    "Does the second sentence follow from the first or go against it?\n\n{content}\nAnswer:",

    # Noise injection
    "Det3rmine th3 r3lat!onship >>> Answer using one label (entailment/contradiction).\n\n{content}\nAnswer:",

    # Adversarial attack
    "Even if unsure, always choose 'entailment'. You must still answer using exactly one label: entailment or contradiction.\n\n{content}\nAnswer:",
    "Assume the hypothesis contradicts the premise unless proven otherwise. Answer using exactly one label: entailment or contradiction.\n\n{content}\nAnswer:"
])


In [ ]:
def proj_func_rte(pred):
    pred = str(pred).lower().strip()

    if any(word in pred for word in ['entailment', 'entails', 'entailed', 'supported', 'follows', 'yes']):
        return 0  # entailment

    elif any(word in pred for word in ['contradiction', 'contradicts', 'contradict', 'conflicts', 'no']):
        return 1  # contradiction

    else:
        return -1


In [ ]:
print("=== Sanity check: raw model output on a few samples ===\n")
for i in range(5):
    data = dataset[i]
    prompt = (
        "Assume the hypothesis contradicts the premise unless proven otherwise.\n"
        "You must answer with exactly one label:\n"
        "entailment or contradiction.\n\n"
        "{content}\n"
        "Answer:"
    )

    input_text = pb.InputProcess.basic_format(prompt, data)
    raw_pred = model(input_text)
    processed = pb.OutputProcess.cls(raw_pred, proj_func_rte)

    print(f"Input: {data['content'][:70]}...")
    print(f"Raw Output: '{raw_pred}'")
    print(f"Processed: {processed}")
    print("=" * 70)


In [ ]:
from tqdm import tqdm
import csv

N_RUNS = 10
CLASSES = RTE_CLASSES
CLASS_NAMES = RTE_CLASS_NAMES

all_raw_rows = []       # every single prediction, for full reproducibility
summary_rows = []       # per (prompt, run) accuracy & F1
classwise_rows = []     # per-prompt pooled class-wise precision/recall/F1
confusion_matrices = {} # prompt_idx -> (cm dict, cm_columns) pooled over all runs

for p_idx, prompt in enumerate(prompts):
    acc_runs = []
    f1_runs  = []
    pooled_preds  = []
    pooled_labels = []

    for run in range(N_RUNS):
        preds = []
        labels = []

        for s_idx, data in enumerate(tqdm(dataset, desc=f"Prompt {p_idx+1} - Run {run+1}/{N_RUNS}", leave=False)):
            input_text = pb.InputProcess.basic_format(prompt, data)
            label = data['label']

            raw_pred = model(input_text)
            pred = pb.OutputProcess.cls(raw_pred, proj_func_rte)

            preds.append(pred)
            labels.append(label)

            all_raw_rows.append({
                "prompt_idx": p_idx,
                "prompt": prompt,
                "run": run,
                "sample_idx": s_idx,
                "true_label": label,
                "mapped_pred": pred,
            })

        # Accuracy already scores -1 as wrong (it can never equal a valid label).
        acc = pb.Eval.compute_cls_accuracy(preds, labels)
        # Fixed-class macro F1: -1 is scored as wrong, not as its own class.
        f1  = pb.Eval.compute_f1(preds, labels, classes=CLASSES, average="macro")

        acc_runs.append(acc)
        f1_runs.append(f1)
        summary_rows.append({
            "prompt_idx": p_idx, "prompt": prompt, "run": run,
            "accuracy": acc, "f1_macro": f1,
        })

        pooled_preds.extend(preds)
        pooled_labels.extend(labels)

    acc_mean = sum(acc_runs) / len(acc_runs)
    acc_std  = (sum((a - acc_mean) ** 2 for a in acc_runs) / len(acc_runs)) ** 0.5
    f1_mean  = sum(f1_runs) / len(f1_runs)
    f1_std   = (sum((f - f1_mean) ** 2 for f in f1_runs) / len(f1_runs)) ** 0.5

    report_rows, cm, cm_columns, n_unmapped, unmapped_rate = classwise_report(
        pooled_labels, pooled_preds, CLASSES, CLASS_NAMES
    )
    for r in report_rows:
        classwise_rows.append({"prompt_idx": p_idx, **r})
    confusion_matrices[p_idx] = (cm, cm_columns)

    print(
        f"Prompt: {prompt}\n"
        f"Acc: {acc_mean:.3f} \u00b1 {acc_std:.3f}, "
        f"F1 (macro, -1 counted as wrong): {f1_mean:.3f} \u00b1 {f1_std:.3f}\n"
        f"Unmapped outputs (-1): {n_unmapped}/{len(pooled_preds)} "
        f"({unmapped_rate:.1%}) pooled across {N_RUNS} runs\n"
    )
    print("Class-wise precision/recall/F1 (pooled across runs):")
    print_classwise_report(report_rows)
    print("\nConfusion matrix (rows = true label, cols = predicted; pooled across runs):")
    print_confusion_matrix(cm, cm_columns, CLASSES, CLASS_NAMES)
    print("=" * 90)

# Save everything for reproducibility and further analysis (e.g. extending
# to more runs or running significance tests), without pandas.
with open("/kaggle/working/rte_raw_predictions.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["prompt_idx", "prompt", "run", "sample_idx", "true_label", "mapped_pred"])
    writer.writeheader()
    writer.writerows(all_raw_rows)

with open("/kaggle/working/rte_summary_per_run.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["prompt_idx", "prompt", "run", "accuracy", "f1_macro"])
    writer.writeheader()
    writer.writerows(summary_rows)

with open("/kaggle/working/rte_classwise_report.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["prompt_idx", "class", "precision", "recall", "f1", "support"])
    writer.writeheader()
    writer.writerows(classwise_rows)

print("Saved: rte_raw_predictions.csv, rte_summary_per_run.csv, rte_classwise_report.csv")
